# Ejecuciones y Resultados

<div style="text-align: justify;">

Este _notebook_ ejecuta las variantes del _NSGA-II_ con los hiperparámetros obtenidos en la parametrización (`Results/Parametrizacion`) sobre los datasets de evaluación (`B1`, `B2`, `B3`), usando **31 semillas** derivadas de la fecha del día y compartidas por todas las variantes. Cada ejecución se guarda de forma incremental (CSV de historial + frente F0 en `.npz`), por lo que una sesión interrumpida se reanuda sin repetir lo ya ejecutado.

</div>

In [1]:
# 1. Clonar el repo
#!git clone https://github.com/BenjaminGonzalezH/Thesis.git /content/Thesis

# 2. Montar Drive
#from google.colab import drive
#drive.mount('/content/drive')

# 3. Copiar las matrices desde Drive al directorio que usa el notebook
import shutil
from pathlib import Path

DRIVE_MATRICES = Path("/content/drive/MyDrive/")                   # <-- ajusta a tu carpeta en Drive
DEST_MATRICES  = Path("/content/Thesis/Datasets/Matrices")        # minúscula: así lo espera el notebook en Colab

DEST_MATRICES.mkdir(parents=True, exist_ok=True)

archivos = list(DRIVE_MATRICES.glob("*.npz"))
for f in archivos:
    shutil.copy2(f, DEST_MATRICES / f.name)

print(f"{len(archivos)} archivos copiados a {DEST_MATRICES}")
print(sorted(p.name for p in DEST_MATRICES.iterdir()))

0 archivos copiados a \content\Thesis\Datasets\Matrices
['A1_S_DB.npz', 'A1_S_DE.npz', 'A2_S_DB.npz', 'A2_S_DB_M.npz', 'A2_S_DE.npz', 'B1_S_DB.npz', 'B1_S_DB_M.npz', 'B1_S_DE.npz', 'B2_S_DB.npz', 'B2_S_DB_M.npz', 'B2_S_DE.npz', 'B3_S_DB.npz', 'clustered_heatmap_BI_A1.png', 'clustered_heatmap_BI_A1_M.png', 'clustered_heatmap_BI_A2.png', 'clustered_heatmap_BI_A2_M.png', 'clustered_heatmap_BI_B1.png', 'clustered_heatmap_BI_B1_M.png', 'clustered_heatmap_BI_B2.png', 'clustered_heatmap_BI_B2_M.png', 'clustered_heatmap_BI_B3.png', 'clustered_heatmap_BI_B3_M.png', 'clustered_heatmap_GE_A1.png', 'clustered_heatmap_GE_A2.png', 'clustered_heatmap_GE_B1.png', 'clustered_heatmap_GE_B2.png', 'clustered_heatmap_GE_B3.png']


## 1. Ambiente y semillas

In [2]:
#######################################################
# MODULO DE PREPARACIÓN DEL AMBIENTE
# Detección de entorno (Colab / Local), rutas del repositorio, respaldo en
# Drive y semillas de las ejecuciones (31, derivadas de la fecha del día).
#######################################################

############################
# Importaciones
############################
import json                                                 # Guardado de las semillas y de las configuraciones.
import sys                                                  # Ajuste de sys.path para importar los módulos de Source Code.
import os                                                   # Llamadas al SO.
import shutil                                               # Copia de archivos entre el disco local y Drive.
from datetime import date                                   # Semilla basada en la fecha del día.
import numpy  as np                                         # Aplicación de operaciones matematicas eficientes.
import pandas as pd                                         # Historial y resumen de resultados.
from pathlib import Path                                    # Aplicar un formato generalizado de direcciones en el script.

############################
# Detección de entorno (Colab / Local)
############################
try:
    import google.colab                                     # Módulo disponible solo dentro de runtimes de Colab.
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    THESIS_DIR = Path("/content/Thesis")
else:
    _cwd = Path(os.getcwd()).resolve()
    THESIS_DIR = next(
        (p for p in (_cwd, *_cwd.parents) if (p / "Datasets").is_dir() and (p / "Source Code").is_dir()),
        None,
    )
    if THESIS_DIR is None:
        raise FileNotFoundError(f"No se encontró la raíz del repositorio (Datasets/ y Source Code/) desde {_cwd}")

MATRICES_DIR = THESIS_DIR / "Datasets" / "Matrices"
print(f"[entorno] {'Colab' if IN_COLAB else 'Local'} → {THESIS_DIR}")

SOURCE_CODE_DIR = THESIS_DIR / "Source Code"
if str(SOURCE_CODE_DIR) not in sys.path:
    sys.path.append(str(SOURCE_CODE_DIR))

############################
# Directorios de resultados y respaldo en Drive
############################
PARAM_DIR   = THESIS_DIR / "Results" / "Parametrizacion"   # Hiperparámetros obtenidos en la parametrización.
RESULTS_DIR = THESIS_DIR / "Results" / "Ejecuciones"       # Resultados de este notebook.
FRENTES_DIR = RESULTS_DIR / "Frentes"                      # Un .npz por ejecución (F0, medoides e historial de HV).
FRENTES_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_RESULTS_DIR = Path("/content/drive/MyDrive/Thesis_Results/Ejecuciones")    # <-- ajusta a tu carpeta en Drive
DRIVE_PARAM_DIR   = Path("/content/drive/MyDrive/Thesis_Results/Parametrizacion")

def _copiar_arbol(origen: Path, destino: Path, restaurar: bool) -> None:
    """Copia recursiva de `origen` a `destino`. Con `restaurar`, no pisa archivos existentes; si no, solo copia lo nuevo o modificado."""
    if not origen.exists():
        return
    for archivo in origen.rglob("*"):
        if not archivo.is_file():
            continue
        objetivo = destino / archivo.relative_to(origen)
        if objetivo.exists() and (restaurar or objetivo.stat().st_mtime >= archivo.stat().st_mtime):
            continue
        objetivo.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(archivo, objetivo)

def sincronizar_drive(restaurar: bool = False) -> None:
    """Local -> Drive (o Drive -> local si `restaurar`). No hace nada fuera de Colab."""
    if not IN_COLAB:
        return
    try:
        if restaurar:
            _copiar_arbol(DRIVE_PARAM_DIR,   PARAM_DIR,   restaurar=True)
            _copiar_arbol(DRIVE_RESULTS_DIR, RESULTS_DIR, restaurar=True)
        else:
            _copiar_arbol(RESULTS_DIR, DRIVE_RESULTS_DIR, restaurar=False)
    except Exception as e:                                  # Un fallo de Drive no debe interrumpir las ejecuciones.
        print(f"[drive] no se pudo {'restaurar' if restaurar else 'respaldar'}: {e}")

sincronizar_drive(restaurar=True)

############################
# Semillas (31, derivadas de la fecha del día AAAAMMDD)
############################
N_SEEDS = 31
SEEDS_FILE = RESULTS_DIR / "seeds_ejecuciones.json"

SEED = int(date.today().strftime("%Y%m%d"))                 # Semilla del día: origen de las 31 semillas.
SEEDS = [int(s) for s in np.random.default_rng(SEED).integers(0, 2**31 - 1, size=N_SEEDS)]

# Persistencia: si ya existen semillas guardadas (sesión previa), se reutilizan para que
# la serie de ejecuciones sea siempre la misma aunque el notebook se reanude otro día.
if SEEDS_FILE.exists():
    with open(SEEDS_FILE, "r", encoding="utf-8") as f:
        _guardado = json.load(f)
    SEED, SEEDS = _guardado["seed"], _guardado["seeds"]
    print(f"[semillas] cargadas desde {SEEDS_FILE.name} (generadas el {_guardado['fecha']})")
else:
    with open(SEEDS_FILE, "w", encoding="utf-8") as f:
        json.dump({"fecha": date.today().isoformat(), "seed": SEED, "seeds": SEEDS}, f, indent=2)
    print(f"[semillas] archivo creado: {SEEDS_FILE.name}")
print(f"[semillas] SEED = {SEED} | {len(SEEDS)} semillas: {SEEDS[:5]} ...")
print(f"[resultados] {RESULTS_DIR}")

[entorno] Local → C:\Users\benja\Desktop\workspace\Thesis
[semillas] cargadas desde seeds_ejecuciones.json (generadas el 2026-11-01)
[semillas] SEED = 20261101 | 31 semillas: [1183181808, 1270388439, 1658144228, 768834435, 978234145] ...
[resultados] C:\Users\benja\Desktop\workspace\Thesis\Results\Ejecuciones


## 2. Código fuente e hiperparámetros

In [3]:
#######################################################
# MODULO DE IMPORTACIÓN DEL CÓDIGO FUENTE E HIPERPARÁMETROS
#######################################################

import solution_encoding                                    # OBJ_FUNCTION_CALLS es un contador global: se reinicia antes de cada ejecución.
from solution_encoding import load_distance_matrix
from LS import build_neighborhood_matrix                    # M_V de MOLS: GE, BI o BOTH.
from nsga2 import run_nsga2
from performance import jaccard_f0_history, jaccard_population_history   # Jaccard de F0 y de la población, por generación.
from nsga2_variants import run_nsga2_MOPR, run_nsga2_PLS, run_nsga2_MOLS
from nsga2_variants_bio import run_nsga2_PNS, run_nsga2_DBS, run_nsga2_MPS

VARIANTES = {
    "NSGA2":       run_nsga2,
    "NSGA2_PLS":   run_nsga2_PLS,
    "NSGA2_MOLS":  run_nsga2_MOLS,
    "NSGA2_DBS":   run_nsga2_DBS,
    "NSGA2_PNS":   run_nsga2_PNS,
    "NSGA2_MPS":   run_nsga2_MPS,
    "NSGA2_MOPR":  run_nsga2_MOPR,
}

############################
# Mejores hiperparámetros de la parametrización
############################
def cargar_mejores_parametros() -> dict:
    """
    Lee `mejores_parametros.json`. Si no existe (la parametrización no llegó a escribirlo),
    se reconstruye desde los estudios de Optuna (`<variante>.db`).
    """
    archivo = PARAM_DIR / "mejores_parametros.json"
    if archivo.exists():
        with open(archivo, "r", encoding="utf-8") as f:
            return json.load(f)

    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    mejores = {}
    for variante in VARIANTES:
        db = PARAM_DIR / f"{variante}.db"
        if not db.exists():
            raise FileNotFoundError(f"Sin parámetros para {variante}: falta {archivo.name} y {db.name} en {PARAM_DIR}")
        estudio = optuna.load_study(study_name=variante, storage=f"sqlite:///{db}")
        mejores[variante] = {"hv_f1": estudio.best_value, "parametros": estudio.best_params}
    print(f"[parámetros] reconstruidos desde los estudios .db ({PARAM_DIR})")
    return mejores

MEJORES = cargar_mejores_parametros()
for v in VARIANTES:
    if v in MEJORES:
        print(f"  {v:<11} HV param = {MEJORES[v]['hv_f1']:.4f} | {MEJORES[v]['parametros']}")
    else:
        print(f"  {v:<11} SIN parámetros (ejecutar Parametrización.ipynb antes de ejecutarla)")

[backend] CuPy detectado → operaciones vectorizadas en GPU.
[backend] CuPy detectado → operaciones vectorizadas en GPU.
  NSGA2       HV param = 14286.6881 | {'pop_size': 111, 'crossover_prob': 0.9, 'mutation_prob': 0.05}
  NSGA2_PLS   HV param = 14294.6857 | {'pop_size': 300, 'crossover_prob': 0.8500000000000001, 'mutation_prob': 0.05, 'pls_neighborhood': 0.18, 'pls_local_budget': 2300}
  NSGA2_MOLS  HV param = 14285.3261 | {'pop_size': 196, 'crossover_prob': 0.8700000000000001, 'mutation_prob': 0.05, 'M_v': 'BOTH', 'mode': 'large', 'mols_local_budget': 9000, 'neighborhood_pct': 29}
  NSGA2_DBS   HV param = 14295.0692 | {'pop_size': 117, 'crossover_prob': 0.89, 'mutation_prob': 0.01, 'mode': 'expresion', 'p_pct': 10}
  NSGA2_PNS   HV param = 14295.1473 | {'pop_size': 168, 'crossover_prob': 0.8300000000000001, 'mutation_prob': 0.05, 'nF': 1}
  NSGA2_MPS   HV param = 14291.6657 | {'pop_size': 151, 'crossover_prob': 0.8600000000000001, 'mutation_prob': 0.03, 'L': 33, 'q_pct': 4}
  NSGA2_

## 3. Cuadro de ejecución

Define qué variantes, datasets y semillas se ejecutan. Los resultados quedan en `Results/Ejecuciones`:

| Archivo | Contenido |
|---|---|
| `historial_ejecuciones.csv` | Registro acumulativo (una fila por ejecución: variante, dataset, semilla, hipervolumen inicial/final, generaciones, llamadas, tiempo, fecha y parámetros). |
| `Frentes/<variante>_<dataset>_<semilla>.npz` | Frente F0 (objetivos y medoides) e historial de hipervolumen de la ejecución. |
| `seeds_ejecuciones.json` | Las 31 semillas usadas. |

Las ejecuciones ya registradas (misma variante, dataset y semilla) se omiten, salvo que `REEJECUTAR = True`; en ese caso se agrega una fila nueva al historial y el resumen usa la más reciente.

In [4]:
#######################################################
# MODULO DE EJECUCIÓN (EN PARALELO)
# Ejecuta variante × dataset × semilla con los mejores hiperparámetros.
# Cada corrida corre en un worker (Source Code/ejecutar.py); el proceso padre
# registra cada resultado apenas llega (CSV de historial) y respalda en Drive.
# Las semillas fijan cada corrida, por lo que el resultado no depende de N_WORKERS.
#######################################################

import time                                                 # Tiempo total.
import multiprocessing                                      # Contexto `spawn` (seguro con numpy/CuPy).
from concurrent.futures import ProcessPoolExecutor, wait, FIRST_COMPLETED
import ejecutar as ejec                                     # Source Code/ejecutar.py: corrida de una variante.

############################
# Configuración del usuario
############################
VARIANTES_A_EJECUTAR = [v for v in VARIANTES if v in MEJORES]  # Solo las ya parametrizadas. Subconjunto, p. ej. ["NSGA2", "NSGA2_PNS"].
DATASETS_A_EJECUTAR  = ["B1", "B2", "B3"]                   # Datasets de evaluación (prefijo B).
SEEDS_A_EJECUTAR     = SEEDS                                # Subconjunto, p. ej. SEEDS[:5] para una prueba.
REEJECUTAR           = False                                # True: repite ejecuciones ya registradas (se agrega al historial).

# Procesos simultáneos. Cada uno carga un dataset (y sus M_V si hay MOLS): la RAM crece
# en proporción. Con GPU (CuPy) todos comparten la misma tarjeta: probar con 2-3 antes de subir.
N_WORKERS = 1 #max(1, min(3, (os.cpu_count() or 2) - 1))

############################
# Constantes del experimento
############################
K                      = 4                                  # Medoides por solución.
MAX_OBJ_FUNCTION_CALLS = 400_000                            # Presupuesto POR ejecución.
HISTORIAL_FILE         = RESULTS_DIR / "historial_ejecuciones.csv"
assert set(VARIANTES_A_EJECUTAR) <= set(VARIANTES), "Variante desconocida en VARIANTES_A_EJECUTAR."

############################
# Historial (solo lo escribe el proceso padre)
############################
def leer_historial() -> pd.DataFrame:
    return pd.read_csv(HISTORIAL_FILE) if HISTORIAL_FILE.exists() else pd.DataFrame()

def registrar(fila: dict) -> None:
    pd.DataFrame([fila]).to_csv(HISTORIAL_FILE, mode="a", header=not HISTORIAL_FILE.exists(), index=False)

def ejecutadas() -> set:
    h = leer_historial()
    return set() if h.empty else set(zip(h["variante"], h["dataset"], h["semilla"]))

############################
# Cuadro de ejecución (ordenado por dataset: cada worker reutiliza el dataset ya cargado)
############################
hechas = set() if REEJECUTAR else ejecutadas()
tareas = [(v, ds, s) for ds in DATASETS_A_EJECUTAR for v in VARIANTES_A_EJECUTAR for s in SEEDS_A_EJECUTAR
          if (v, ds, s) not in hechas]
total  = len(VARIANTES_A_EJECUTAR) * len(DATASETS_A_EJECUTAR) * len(SEEDS_A_EJECUTAR)
print(f"[plan] {total} ejecuciones en total | {total - len(tareas)} ya registradas | {len(tareas)} pendientes | {N_WORKERS} procesos")

ERRORES, terminadas, t_ini = {}, 0, time.time()
if tareas:
    ejecutor = ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=multiprocessing.get_context("spawn"))
    try:
        futuros = {
            ejecutor.submit(ejec.ejecutar, v, ds, s, MEJORES[v]["parametros"],
                            MATRICES_DIR, FRENTES_DIR, K, MAX_OBJ_FUNCTION_CALLS): (v, ds, s)
            for v, ds, s in tareas
        }
        pendientes = set(futuros)
        while pendientes:
            terminados, pendientes = wait(pendientes, return_when=FIRST_COMPLETED)
            for f in terminados:
                v, ds, s = futuros[f]
                if f.exception() is not None:
                    ERRORES[(v, ds, s)] = f.exception()
                    print(f"[ERROR] {v} {ds} semilla={s}: {f.exception()!r}")
                    continue
                fila = f.result()
                registrar(fila)
                terminadas += 1
                print(f"[{total - len(tareas) + terminadas:>4}/{total}] {v:<11} {ds} semilla={s:<11} "
                      f"HV F0 = {fila['hv_final']:.4f} | |F0| = {fila['tam_f0']:>3} | {fila['segundos'] / 60:.1f} min")
            sincronizar_drive()                             # Respaldo en Drive (solo Colab; solo el proceso padre).
    except KeyboardInterrupt:
        print("[interrumpido] se detienen los procesos; lo ya registrado se conserva y se reanuda al volver a ejecutar la celda.")
        for p in list(getattr(ejecutor, "_processes", {}).values()):
            p.terminate()
        raise
    finally:
        ejecutor.shutdown(wait=False, cancel_futures=True)

if ERRORES:
    print(f"[aviso] {len(ERRORES)} ejecuciones con error; vuelve a ejecutar la celda para reintentarlas.")
print(f"\n[fin] {terminadas} ejecuciones completadas en {(time.time() - t_ini) / 60:.1f} min")


[plan] 651 ejecuciones en total | 558 ya registradas | 93 pendientes | 1 procesos
[ 559/651] NSGA2_MPS   B1 semilla=1183181808  HV F0 = 14271.6842 | |F0| =  18 | 0.4 min
[ 560/651] NSGA2_MPS   B1 semilla=1270388439  HV F0 = 14276.1363 | |F0| =  16 | 1.2 min
[ 561/651] NSGA2_MPS   B1 semilla=1658144228  HV F0 = 14271.9799 | |F0| =  15 | 0.4 min
[ 562/651] NSGA2_MPS   B1 semilla=768834435   HV F0 = 14272.8114 | |F0| =  19 | 0.7 min
[ 563/651] NSGA2_MPS   B1 semilla=978234145   HV F0 = 14271.9690 | |F0| =  14 | 0.5 min
[ 564/651] NSGA2_MPS   B1 semilla=1039227980  HV F0 = 14270.2831 | |F0| =  15 | 0.4 min
[ 565/651] NSGA2_MPS   B1 semilla=245761548   HV F0 = 14270.7430 | |F0| =  13 | 0.5 min
[ 566/651] NSGA2_MPS   B1 semilla=2118637595  HV F0 = 14273.1333 | |F0| =  18 | 0.7 min
[ 567/651] NSGA2_MPS   B1 semilla=358181644   HV F0 = 14271.2072 | |F0| =  18 | 0.5 min
[ 568/651] NSGA2_MPS   B1 semilla=291931378   HV F0 = 14269.3137 | |F0| =  12 | 1.2 min
[ 569/651] NSGA2_MPS   B1 semilla=2116

## 4. Resultados

Resumen del hipervolumen final de F0 por variante y dataset (usa la ejecución más reciente de cada combinación).

In [5]:
#######################################################
# MODULO DE RESUMEN DE RESULTADOS
#######################################################

historial = leer_historial()
if historial.empty:
    print("Aún no hay ejecuciones registradas.")
else:
    ultimas = historial.sort_values("fecha").drop_duplicates(["variante", "dataset", "semilla"], keep="last")

    resumen = (ultimas.groupby(["dataset", "variante"])
               .agg(n=("hv_final", "size"), hv_media=("hv_final", "mean"), hv_desv=("hv_final", "std"),
                    hv_min=("hv_final", "min"), hv_max=("hv_final", "max"),
                    tam_f0_medio=("tam_f0", "mean"), minutos_medio=("segundos", lambda s: s.mean() / 60))
               .round(4))
    resumen.to_csv(RESULTS_DIR / "resumen_resultados.csv")
    sincronizar_drive()
    display(resumen)

    # Matriz semilla × variante (HV final) por dataset: base para las pruebas estadísticas.
    for nombre_ds, g in ultimas.groupby("dataset"):
        print(f"\n{nombre_ds}: HV final por semilla")
        display(g.pivot(index="semilla", columns="variante", values="hv_final").round(4))

n    hv_media  hv_desv      hv_min      hv_max  \
dataset variante                                                      
B1      NSGA2       31  14267.2410   2.6694  14261.9322  14272.5401   
        NSGA2_DBS   31  14273.2595   0.8443  14271.5613  14275.2789   
        NSGA2_MOLS  31  14267.5226   4.7826  14255.8694  14275.3376   
        NSGA2_MOPR  31  14266.5300   3.2155  14261.8673  14272.6910   
        NSGA2_MPS   31  14271.6395   2.0524  14264.4319  14276.1363   
        NSGA2_PLS   31  14273.4418   1.4488  14270.6413  14277.2748   
        NSGA2_PNS   31  14273.3417   1.5494  14270.6034  14277.6921   
B2      NSGA2       31  14287.7651   2.2469  14283.7740  14291.7979   
        NSGA2_DBS   31  14293.6160   1.6637  14287.2681  14294.9554   
        NSGA2_MOLS  31  14284.8178   3.4675  14275.1294  14290.8818   
        NSGA2_MOPR  31  14286.4480   2.1473  14279.7220  14290.6965   
        NSGA2_MPS   31  14290.6607   2.9243  14284.5126  14294.7254   
        NSGA2_PLS   31  14292.3949   1.6573  14288.6923  14295.0171   
        NSGA2_PNS   31  14292.1990   1.5737  14288.7617  14294.9579   
B3      NSGA2       31  14265.5930   1.6519  14262.0668  14268.9605   
        NSGA2_DBS   31  14271.4494   1.9266  14266.1308  14275.1881   
        NSGA2_MOLS  31  14269.0627   2.0167  14263.7935  14272.1616   
        NSGA2_MOPR  31  14265.3117   1.7080  14260.5036  14268.2500   
        NSGA2_MPS   31  14268.2539   2.0890  14263.4760  14272.5331   
        NSGA2_PLS   31  14270.7325   1.6657  14266.3447  14273.2227   
        NSGA2_PNS   31  14271.7889   1.4413  14269.1868  14274.3340   

                    tam_f0_medio  minutos_medio  
dataset variante                                 
B1      NSGA2            14.0000         2.7483  
        NSGA2_DBS        19.4516         0.7990  
        NSGA2_MOLS       17.5806         0.3576  
        NSGA2_MOPR       14.8710         3.6909  
        NSGA2_MPS        15.8710         0.5781  
        NSGA2_PLS        20.4194         1.1328  
        NSGA2_PNS        19.5161         3.1097  
B2      NSGA2            15.9677         2.7374  
        NSGA2_DBS        23.6129         0.8256  
        NSGA2_MOLS       24.1613         0.4477  
        NSGA2_MOPR       16.2581         3.4946  
        NSGA2_MPS        27.2581         1.3803  
        NSGA2_PLS        25.2258         1.0753  
        NSGA2_PNS        27.0323         3.2292  
B3      NSGA2            26.1290         6.6609  
        NSGA2_DBS        29.1613         0.7714  
        NSGA2_MOLS       33.8710         0.8076  
        NSGA2_MOPR       27.2258         4.6052  
        NSGA2_MPS        24.3871         2.9862  
        NSGA2_PLS        35.3226         3.2354  
        NSGA2_PNS        36.1613         4.6906


B1: HV final por semilla


variante,NSGA2,NSGA2_DBS,NSGA2_MOLS,NSGA2_MOPR,NSGA2_MPS,NSGA2_PLS,NSGA2_PNS
semilla,,,,,,,
2241395,14262.9398,14273.4867,14263.6049,14270.6990,14271.6459,14273.0461,14273.9820
149968717,14268.1621,14273.0523,14264.9560,14262.7217,14271.8658,14273.2028,14273.1813
245761548,14265.4540,14272.8337,14269.2541,14262.1620,14270.7430,14271.9490,14277.6921
291931378,14269.6565,14273.2387,14263.3395,14272.6910,14269.3137,14273.5282,14272.1865
346868666,14265.8505,14274.1555,14271.1147,14261.8673,14270.5950,14276.6944,14275.0929
358181644,14269.2925,14273.0507,14269.6168,14266.4289,14271.2072,14274.5984,14270.6034
398383450,14266.1480,14273.7159,14273.5728,14265.4463,14274.4324,14274.9755,14273.5628
463504515,14270.9539,14274.2580,14273.1398,14270.9736,14271.1740,14273.2269,14272.4453
536623952,14264.5637,14273.7752,14255.9442,14268.9158,14270.5903,14274.0036,14273.0516



B2: HV final por semilla


variante,NSGA2,NSGA2_DBS,NSGA2_MOLS,NSGA2_MOPR,NSGA2_MPS,NSGA2_PLS,NSGA2_PNS
semilla,,,,,,,
2241395,14285.0798,14294.3917,14286.1688,14285.0990,14287.2684,14292.5415,14292.4037
149968717,14288.5353,14294.4156,14284.6373,14287.0343,14288.2005,14294.5477,14290.9461
245761548,14284.9465,14294.9050,14290.8818,14285.7080,14294.7222,14290.9097,14294.2745
291931378,14288.2032,14294.4233,14286.3543,14287.2544,14289.0741,14293.2833,14293.3053
346868666,14287.1502,14294.9550,14285.6261,14279.7220,14292.8566,14293.3014,14290.9607
358181644,14287.4810,14287.2681,14279.5282,14289.0326,14291.3668,14292.4726,14288.7617
398383450,14287.5911,14292.1460,14287.4900,14289.2677,14292.0351,14292.1280,14291.9778
463504515,14286.5674,14294.6280,14286.3167,14286.0687,14294.7254,14292.0018,14292.1023
536623952,14290.1017,14294.6272,14278.0649,14285.1705,14293.5058,14292.2126,14292.8482



B3: HV final por semilla


variante,NSGA2,NSGA2_DBS,NSGA2_MOLS,NSGA2_MOPR,NSGA2_MPS,NSGA2_PLS,NSGA2_PNS
semilla,,,,,,,
2241395,14268.9010,14270.6599,14266.6796,14265.8790,14270.2710,14270.2130,14269.6662
149968717,14266.4358,14271.3975,14269.8446,14263.2960,14268.9077,14268.7547,14274.2644
245761548,14263.2455,14266.1308,14267.9737,14264.5356,14266.4345,14271.9453,14272.7903
291931378,14266.0823,14271.3960,14271.0439,14265.7650,14266.8121,14269.0424,14271.8715
346868666,14265.4603,14271.7539,14272.1616,14266.2612,14270.2437,14271.4410,14273.6694
358181644,14264.8696,14269.8746,14271.0076,14263.3889,14265.1051,14271.8942,14272.5418
398383450,14265.7442,14272.8741,14268.6495,14260.5036,14269.3144,14270.9944,14272.1983
463504515,14264.1717,14273.2980,14268.3548,14266.6679,14270.1164,14267.3249,14274.3139
536623952,14264.4260,14270.4078,14271.9591,14265.3711,14271.4719,14272.9871,14273.2669
